# Project 8 — eval noise & energy regression

Adapted from `chreissel/neutrino_project/notebooks/eval-noise.ipynb` for the
current TimeSeriesPhysics layout.

- Model: `tasks.Project8.project8_regression.Project8Regression`
  (encoder = `models.s4d.S4Model`, output = [mu, log_var] for energy_eV).
- DataModule: `dataloader.project8_dataloader.Project8DataModule`.
  Yields `(X, var)` — `X` is the time series when `freq_transform=None`,
  the FFT (real/imag) when `freq_transform='fft'`.

We instantiate the DataModule manually here (Lightning DataModules aren't
saved in checkpoints), so adjust the `data_kwargs` block to match the run
you're evaluating.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
# Make src/ packages importable when running the notebook in-place.
_repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.join(_repo_root, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(_repo_root, 'src'))

import numpy as np
import torch
from scipy import signal
import matplotlib.pyplot as plt

from tasks.Project8.project8_regression import Project8Regression
from dataloader.project8_dataloader import Project8DataModule

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
CHECKPOINT = '/n/holystore01/LABS/iaifi_lab/Lab/creissel/neutrino_mass/trainings/project8_s4d_regression_energy_gaussiannll/best.ckpt'

In [ ]:
model = Project8Regression.load_from_checkpoint(CHECKPOINT, map_location=device)
model = model.to(device).eval()

In [ ]:
# Match the training config; use freq_transform='fft' for inference,
# and set norm stats to whatever was used at training time.
data_kwargs = dict(
    train_dir='data/Project8Sim/hdf5/train',
    val_dir='data/Project8Sim/hdf5/val',
    test_dir='data/Project8Sim/hdf5/test',
    inputs=['output_ts_I', 'output_ts_Q'],
    variables=['energy_eV'],
    cutoff=8192,
    norm=True,
    noise_type='cav',
    freq_transform='fft',
    batch_size=128,
    num_workers=0,
)
dm = Project8DataModule(**data_kwargs)
dm.setup('test')
print('mu  =', dm.mu)
print('std =', dm.stds)

In [ ]:
# PSD sanity check — needs raw time series, so build a parallel DataModule
# with freq_transform=None.
dm_ts = Project8DataModule(**{**data_kwargs, 'freq_transform': None})
dm_ts.setup('test')

for ts, _ in dm_ts.test_dataloader():
    for i in range(10):
        f, pxx = signal.periodogram(ts[i, :, 0].numpy(), fs=403e6)
        plt.plot(f, pxx)
    break
plt.yscale('log')
plt.ylim(1e-10, 1e-5)
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD')
plt.title('Test-set I-channel periodograms');

In [ ]:
energy_ind = dm.hparams.variables.index('energy_eV')

In [ ]:
# Inference loop. The model output is (mu, log_var) per event for energy_eV.
# We also compute simple PSD-derived event features (max / summed power)
# straight from the time series, mirroring the upstream notebook.
max_powers = []
summed_powers = []
true_z = []
pred_mu_z = []
pred_logvar = []

ts_loader = iter(dm_ts.test_dataloader())
with torch.no_grad():
    for x, var in dm.test_dataloader():
        out = model(x.to(device)).cpu().numpy()
        pred_mu_z.append(out[:, 0])
        pred_logvar.append(out[:, 1])
        true_z.append(var.numpy()[:, 0])

        ts, _ = next(ts_loader)
        for i in range(ts.shape[0]):
            f_i, pxx_i = signal.periodogram(ts[i, :, 0].numpy(), fs=403e6)
            f_q, pxx_q = signal.periodogram(ts[i, :, 1].numpy(), fs=403e6)
            pxx_prod = pxx_i * pxx_i + pxx_q * pxx_q
            noise_sum = np.sum(pxx_prod[:400])  # signal-free band
            normed_pxx = pxx_prod / (noise_sum + 1e-30)
            max_powers.append(normed_pxx.max())
            summed_powers.append(normed_pxx.sum())

true_z = np.concatenate(true_z)
pred_mu_z = np.concatenate(pred_mu_z)
pred_logvar = np.concatenate(pred_logvar)
max_powers = np.asarray(max_powers)
summed_powers = np.asarray(summed_powers)

In [ ]:
# Undo z-score normalisation (single target, so mu/stds are scalars).
mu0, std0 = float(dm.mu[energy_ind]), float(dm.stds[energy_ind])
pred_energy = pred_mu_z * std0 + mu0
true_energy = true_z * std0 + mu0
pred_sigma = np.exp(0.5 * pred_logvar) * std0  # in eV
energy_errs = true_energy - pred_energy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(energy_errs, bins=80)
axes[0].set_xlabel('true - pred  [eV]')
axes[0].set_ylabel('events')
axes[0].set_title(f'residual: mean={energy_errs.mean():.2f}, std={energy_errs.std():.2f}')

axes[1].scatter(true_energy, pred_energy, s=2, alpha=0.4)
lo, hi = true_energy.min(), true_energy.max()
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1)
axes[1].set_xlabel('true energy [eV]')
axes[1].set_ylabel('predicted energy [eV]')
axes[1].set_title('regression scatter')

axes[2].scatter(np.abs(energy_errs), pred_sigma, s=2, alpha=0.4)
axes[2].set_xlabel('|residual| [eV]')
axes[2].set_ylabel('predicted sigma [eV]')
axes[2].set_title('GaussianNLL calibration')

fig.tight_layout()

In [ ]:
# Cross-check: residual vs simple PSD-derived event features.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(max_powers, np.abs(energy_errs), s=2, alpha=0.4)
axes[0].set_xlabel('max normed PSD')
axes[0].set_ylabel('|residual| [eV]')
axes[0].set_xscale('log')

axes[1].scatter(summed_powers, np.abs(energy_errs), s=2, alpha=0.4)
axes[1].set_xlabel('summed normed PSD')
axes[1].set_ylabel('|residual| [eV]')
axes[1].set_xscale('log')
fig.tight_layout()